In [ ]:
import os
import numpy as np
import torch
import matplotlib.pyplot as plt
%cd nanoVLM
import os
import numpy as np
import torch
from PIL import Image
import matplotlib.pyplot as plt

import models.config as config
from models.vision_language_model_action import VisionLanguageActionModel
from minigrid.envs.empty import EmptyEnv
from data.processors import get_image_processor, get_image_string
from data.emptyenv_action_dataset import DEFAULT_PROMPT

ID2ACTION = {0: "left", 1: "right", 2: "forward"}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# общий конфиг (одинаковый для всех прогонов)
vlm_cfg = config.VLMConfig()
vlm_cfg.max_img_size = 512


# ---- checkpoint dirs (как ты дал) ----
base_ckpt_dir = "/teamspace/studios/this_studio/nanoVLM-action/nanoVLM/checkpoints_emptyenv_action/base_sft/step_132"
base_ckpt_dir_grpo = "/teamspace/studios/this_studio/nanoVLM-action/nanoVLM/checkpoints_emptyenv_action/grpo/it_2"

common_eval_kwargs = dict(
    num_episodes=50,
    max_steps=125,
    seed=123,
    temperature=1.4,
    greedy=False,
    render=False,
)

# общий image_processor (одинаковый для всех прогонов)
image_processor = get_image_processor(
    vlm_cfg.max_img_size,
    vlm_cfg.vit_img_size,
    vlm_cfg.resize_to_max_side_len
)

@torch.no_grad()
def policy_action_from_rgb(model, rgb, prompt=DEFAULT_PROMPT, temperature=1.0, greedy=False):
    """
    temperature > 1.0  -> more exploration
    temperature < 1.0  -> more greedy
    greedy=True        -> argmax regardless of temperature
    """
    tokenizer = model.tokenizer
    device = next(model.parameters()).device

    img = Image.fromarray(rgb).convert("RGB")
    processed_image, splitted_image_count = image_processor(img)

    messages = [{"role": "user", "content": prompt}]
    image_string = get_image_string(tokenizer, [splitted_image_count], vlm_cfg.mp_image_token_length)
    messages[0]["content"] = image_string + messages[0]["content"]

    conv = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_special_tokens=False,
        return_dict=True,
    )

    input_ids = torch.tensor(conv["input_ids"], dtype=torch.long).unsqueeze(0).to(device)
    attention_mask = torch.tensor(conv["attention_mask"], dtype=torch.long).unsqueeze(0).to(device)
    images = [[processed_image]]  # batch size 1

    logits, _ = model(
        input_ids=input_ids,
        images=images,
        attention_mask=attention_mask,
        action_labels=None
    )  # logits: [1, 3]
    logits = logits[0]  # [3]

    if greedy:
        probs = torch.softmax(logits, dim=-1)
        action = int(torch.argmax(probs).item())
        return action, probs.detach().cpu().numpy()

    T = float(temperature)
    if T <= 0:
        raise ValueError("temperature must be > 0")

    probs = torch.softmax(logits / T, dim=-1)
    action = int(torch.multinomial(probs, num_samples=1).item())
    return action, probs.detach().cpu().numpy()


def run_eval(model, num_episodes=20, sizes=[6, 7], max_steps=200, seed=0,
             temperature=1.0, greedy=False, render=False):
    successes = 0
    returns = []
    lengths = []

    rng = np.random.default_rng(seed)

    for ep in range(num_episodes):
        size = int(rng.choice(sizes))
        env = EmptyEnv(size=size, agent_start_pos=None, render_mode="rgb_array")
        obs, info = env.reset(seed=seed + ep)

        env.agent_dir = int(rng.integers(0, 4))

        ep_ret = 0.0
        ep_len = 0
        terminated = False
        truncated = False

        while not (terminated or truncated):
            rgb = env.render()

            action, probs = policy_action_from_rgb(
                model,
                rgb,
                temperature=temperature,
                greedy=greedy
            )

            obs, reward, terminated, truncated, info = env.step(action)
            ep_ret += float(reward)
            ep_len += 1

            if render:
                print(f"ep={ep} t={ep_len} a={ID2ACTION[action]} probs={np.round(probs,3)}")

            if ep_len >= max_steps:
                break

        success = 1 if ep_ret > 0 else 0
        successes += success
        returns.append(ep_ret)
        lengths.append(ep_len)

        print(f"[EP {ep}] size={size} success={success} return={ep_ret:.2f} len={ep_len}")

        env.close()

    metrics = {
        "success_rate": successes / num_episodes,
        "avg_return": float(np.mean(returns)),
        "avg_length": float(np.mean(lengths)),
    }
    return metrics

def load_model_from_dir(ckpt_dir: str):
    model = VisionLanguageActionModel.from_pretrained(ckpt_dir)
    model.to(device)
    model.eval()
    return model

def eval_per_size(model, sizes=(11,12,13), **kwargs):
    """
    Возвращает dict:
      results[size] = {"success_rate":..., "avg_return":..., "avg_length":...}
    Каждый size прогоняем отдельно (sizes=[size]).
    """
    results = {}
    for s in sizes:
        metrics = run_eval(model, sizes=[int(s)], **kwargs)
        results[int(s)] = metrics
    return results

# ---- load both models ----
model_sft = load_model_from_dir(base_ckpt_dir)
model_grpo = load_model_from_dir(base_ckpt_dir_grpo)

# ---- run eval separately for each size ----
sizes = [11, 12, 13, 14]
res_sft = eval_per_size(model_sft, sizes=sizes, **common_eval_kwargs)
res_grpo = eval_per_size(model_grpo, sizes=sizes, **common_eval_kwargs)

# ---- pack for plotting ----
sr_sft   = [res_sft[s]["success_rate"] for s in sizes]
sr_grpo  = [res_grpo[s]["success_rate"] for s in sizes]
ret_sft  = [res_sft[s]["avg_return"]   for s in sizes]
ret_grpo = [res_grpo[s]["avg_return"]  for s in sizes]

print("SFT:", res_sft)
print("GRPO:", res_grpo)

# ---- plotting: grouped bar charts ----
x = np.arange(len(sizes))
width = 0.35

# 1) Success rate histogram
plt.figure(figsize=(8,5))
plt.bar(x - width/2, sr_sft,  width, label="SFT")
plt.bar(x + width/2, sr_grpo, width, label="SFT + GRPO")
plt.xticks(x, [str(s) for s in sizes])
plt.xlabel("Size")
plt.ylabel("Success rate")
plt.title("Success rate by size (50 episodes each, max_steps=150)")
plt.grid(True, axis="y")
plt.legend()
plt.tight_layout()
plt.show()

# 2) Avg return histogram
plt.figure(figsize=(8,5))
plt.bar(x - width/2, ret_sft,  width, label="SFT")
plt.bar(x + width/2, ret_grpo, width, label="SFT + GRPO")
plt.xticks(x, [str(s) for s in sizes])
plt.xlabel("Size")
plt.ylabel("Avg return")
plt.title("Avg return by size (50 episodes each, max_steps=125)")
plt.grid(True, axis="y")
plt.legend()
plt.tight_layout()
plt.show()

/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/IPython/core/magics/osm.py:393: UserWarning: using bookmarks requires you to install the `pickleshare` library.
  bkms = self.shell.db.get('bookmarks', {})


[Errno 2] No such file or directory: 'nanoVLM'
/teamspace/studios/this_studio/nanoVLM-action/nanoVLM
Resize to max side len: True


[EP 0] size=11 success=0 return=0.00 len=125
[EP 1] size=11 success=1 return=0.99 len=5
[EP 2] size=11 success=0 return=0.00 len=125
[EP 3] size=11 success=1 return=0.85 len=82
